---
title: 04 Link Watch Data To Metadata, Exposure Context, And Problematic-View Labels
---

This notebook links cleaned donated YouTube watch data to scraped YouTube metadata and adds event-level context markers: whether a watch closely followed a YouTube search, whether the watched video came from a channel the participant had subscribed to, whether the video is classified as a Short or Long, whether the watch occurred during the configured daytime or nighttime window, and whether a metadata error has been manually categorized as problematic.

The output is an event-level table: each row is still one watch event from the donated history, but it now carries video-level metadata, raw scrape errors, strict search-associated and subscribed-channel flags, local-time day/night labels, Shorts/Longs labels, and transparent problematic-view labels. This creates a shared data foundation for later notebooks on ratios and unavailable or problematic views.

The notebook uses the cleaned files created by the previous notebooks:

- `outputs/tables/video_histories.csv`
- `outputs/tables/meta_data.csv`
- `outputs/tables/search_query_histories.csv`
- `data/mock_takeout/*/YouTube and YouTube Music/subscriptions/subscriptions.csv`
- `data/manual/unavailable_content_classification_filled.csv`

It writes:

- `outputs/tables/video_histories_enriched.csv`
- `outputs/tables/metadata_linkage_report.csv`
- `outputs/tables/subscription_histories.csv`
- `outputs/tables/exposure_linkage_report.csv`
- `outputs/tables/error_types_by_video.csv`


In [ ]:
from pathlib import Path
import re

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)


## Locate Project Files

The notebook is designed to run from the project root or from inside the `scripts` folder. Paths are kept relative to the Quarto project so rendered notebooks do not expose private local folders.

In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        has_quarto_project = (candidate / "_quarto.yml").exists()
        has_output_folder = (candidate / "outputs" / "tables").exists()
        if has_quarto_project and has_output_folder:
            return candidate
    raise FileNotFoundError(
        "Could not find the Quarto project root. Run this notebook from "
        "inside youtube_donation_dsa_method."
    )


PROJECT_ROOT = find_project_root()
MOCK_TAKEOUT_DIR = PROJECT_ROOT / "data" / "mock_takeout"
MANUAL_DIR = PROJECT_ROOT / "data" / "manual"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

VIDEO_HISTORIES_PATH = OUTPUT_DIR / "video_histories.csv"
META_DATA_PATH = OUTPUT_DIR / "meta_data.csv"
PUBLIC_DEMO_META_DATA_PATH = PROJECT_ROOT / "data" / "demo_metadata" / "meta_data_public.csv"
ACTIVE_META_DATA_PATH = META_DATA_PATH if META_DATA_PATH.exists() else PUBLIC_DEMO_META_DATA_PATH
SEARCH_QUERY_HISTORIES_PATH = OUTPUT_DIR / "search_query_histories.csv"
MANUAL_CLASSIFICATION_PATH = MANUAL_DIR / "unavailable_content_classification_filled.csv"
ENRICHED_VIDEO_HISTORIES_PATH = OUTPUT_DIR / "video_histories_enriched.csv"
METADATA_LINKAGE_REPORT_PATH = OUTPUT_DIR / "metadata_linkage_report.csv"
SUBSCRIPTION_HISTORIES_PATH = OUTPUT_DIR / "subscription_histories.csv"
EXPOSURE_LINKAGE_REPORT_PATH = OUTPUT_DIR / "exposure_linkage_report.csv"
ERROR_TYPES_BY_VIDEO_PATH = OUTPUT_DIR / "error_types_by_video.csv"

print(f"Project folder: {PROJECT_ROOT.name}")
print(f"Watch input: {VIDEO_HISTORIES_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Metadata input: {ACTIVE_META_DATA_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Search input: {SEARCH_QUERY_HISTORIES_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Manual classification input: {MANUAL_CLASSIFICATION_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print("Subscription input: data/mock_takeout/*/YouTube and YouTube Music/subscriptions/subscriptions.csv")
print(f"Output folder: {OUTPUT_DIR.relative_to(PROJECT_ROOT).as_posix()}")


## Configure Event Flags

These settings control event-level labels. The mock data represents Danish users, so the default timezone is `Europe/Copenhagen`. The original day/night distinction treated `06:00-23:59` as day and `00:00-05:59` as night; the boundaries are configurable here.

In [ ]:
ANALYSIS_TIMEZONE = "Europe/Copenhagen"
DAYTIME_START_HOUR = 6
DAYTIME_END_HOUR = 24
SHORT_MAX_DURATION_SECONDS = 180
SHORT_MAX_ASPECT_RATIO = 1


def validate_hour_boundary(value, label, allow_24=False):
    upper = 24 if allow_24 else 23
    if not isinstance(value, int) or value < 0 or value > upper:
        raise ValueError(f"{label} must be an integer from 0 to {upper}.")


validate_hour_boundary(DAYTIME_START_HOUR, "DAYTIME_START_HOUR")
validate_hour_boundary(DAYTIME_END_HOUR, "DAYTIME_END_HOUR", allow_24=True)
if DAYTIME_START_HOUR == DAYTIME_END_HOUR:
    raise ValueError("DAYTIME_START_HOUR and DAYTIME_END_HOUR must define a non-empty interval.")


def hour_is_in_interval(hours, start_hour, end_hour):
    if start_hour < end_hour:
        return hours.ge(start_hour) & hours.lt(end_hour)
    return hours.ge(start_hour) | hours.lt(end_hour)


configuration = pd.DataFrame(
    {
        "setting": [
            "ANALYSIS_TIMEZONE",
            "DAYTIME_START_HOUR",
            "DAYTIME_END_HOUR",
            "SHORT_MAX_DURATION_SECONDS",
            "SHORT_MAX_ASPECT_RATIO",
        ],
        "value": [
            ANALYSIS_TIMEZONE,
            DAYTIME_START_HOUR,
            DAYTIME_END_HOUR,
            SHORT_MAX_DURATION_SECONDS,
            SHORT_MAX_ASPECT_RATIO,
        ],
    }
)

configuration


## Configure Problematic-View Classification

The raw scraper error is kept unchanged, but the original workflow also grouped similar error messages before manual classification. These settings make that process visible and editable: the regular expressions normalize common `yt_dlp` messages, the alias map handles known wording variants, and the manual CSV assigns each generalized message to a category and criticality code.


In [ ]:
ACCOUNT_TERMINATED_MESSAGE = (
    "Video unavailable. This video is no longer available because the YouTube account "
    "associated with this video has been terminated."
)
ACCOUNT_TERMINATED_IS_PROBLEMATIC = True

ASSESSMENT_MAP = {
    "Y": "Problematic",
    "N": "Not problematic",
    "Platform external legal action": "Removed due to external legal rights",
}

# The current mock scrape sometimes produces this shorter wording. We map it to
# the existing manual category rather than inventing a new classification row.
ERROR_CLASSIFICATION_MESSAGE_ALIASES = {
    "This video is not available": "Video unavailable. This video is not available",
}

ANSI_ESCAPE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")
COUNTRY_PREFIX = "The uploader has not made this video available in your country"
ERROR_NORMALIZATION_RULES = [
    (r"(copyright claim)\s+by.*", r"\1", re.IGNORECASE),
    (
        r"Video unavailable\. This video contains content from .*?,.*?blocked it(?: in your country)? on copyright grounds",
        "Video unavailable. This video contains content from copyright owner(s) who have blocked it on copyright grounds",
        re.IGNORECASE,
    ),
    (r"Parse error.*", "Some sort of parse error", re.IGNORECASE),
    (
        r"This video is available to this channel's members on level: .*? \(or any higher level\).*",
        "Members-only video",
        re.IGNORECASE,
    ),
    (r"^.*?\[youtube\]\s+.*?:\s*", "", 0),
    (
        r"Video unavailable\. This video contains content from .*?\. It is not available in your country\.",
        "Video unavailable in your country.",
        re.IGNORECASE,
    ),
    (
        r"ERROR: Unable to download video subtitles for '.*?': ERROR:.*",
        "Unable to download subtitles",
        re.IGNORECASE,
    ),
    (r"Sign in to confirm your age\..*", "Age-restricted video. Sign-in required", re.IGNORECASE),
    (
        r"(Video unavailable\. This video contains content from )[^.]+(\. It is not available\.?)",
        r"\1creator\2",
        re.IGNORECASE,
    ),
]


def normalize_error_message(raw_error, video_id):
    if pd.isna(raw_error):
        return raw_error

    text = str(raw_error)
    if pd.notna(video_id):
        text = text.replace(str(video_id), "video_id")

    text = ANSI_ESCAPE.sub("", text)
    text = text.replace("ERROR: [youtube] video_id:", "")
    text = text.strip()

    for pattern, replacement, flags in ERROR_NORMALIZATION_RULES:
        text = re.sub(pattern, replacement, text, flags=flags).strip()

    if COUNTRY_PREFIX in text:
        text = COUNTRY_PREFIX

    return text.strip()


problematic_view_configuration = pd.DataFrame(
    {
        "setting": [
            "MANUAL_CLASSIFICATION_PATH",
            "ACCOUNT_TERMINATED_IS_PROBLEMATIC",
            "ERROR_NORMALIZATION_RULE_COUNT",
            "ERROR_CLASSIFICATION_ALIAS_COUNT",
        ],
        "value": [
            MANUAL_CLASSIFICATION_PATH.relative_to(PROJECT_ROOT).as_posix(),
            ACCOUNT_TERMINATED_IS_PROBLEMATIC,
            len(ERROR_NORMALIZATION_RULES),
            len(ERROR_CLASSIFICATION_MESSAGE_ALIASES),
        ],
    }
)

problematic_view_configuration


## Load Watch Events And Metadata

Watch history and metadata have different units of analysis. The watch table is event-level: the same video may appear multiple times for one or more participants. The metadata table is video-level: each video ID should appear once after deduplication.

The enriched output keeps the watch-event unit because later analyses need participant IDs and timestamps.

In [ ]:
for path, label in [
    (VIDEO_HISTORIES_PATH, "video_histories.csv"),
    (ACTIVE_META_DATA_PATH, "metadata table"),
    (SEARCH_QUERY_HISTORIES_PATH, "search_query_histories.csv"),
    (MANUAL_CLASSIFICATION_PATH, "unavailable_content_classification_filled.csv"),
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {label}: {path.relative_to(PROJECT_ROOT).as_posix()}"
        )


video_histories = pd.read_csv(
    VIDEO_HISTORIES_PATH,
    dtype={"Participant ID": "string", "video_id": "string", "channel_id": "string"},
    low_memory=False,
)

# Full descriptions and subtitle text can be very large. They stay in meta_data.csv,
# while this event-level handoff keeps only fields needed by the next analysis steps.
large_text_metadata_columns = {"description", "subtitles", "requested_subtitles"}
meta_data = pd.read_csv(
    ACTIVE_META_DATA_PATH,
    dtype={"id": "string", "channel_id": "string"},
    usecols=lambda column: column not in large_text_metadata_columns,
    low_memory=False,
)

input_summary = pd.DataFrame(
    {
        "dataset": ["video_histories", "meta_data"],
        "rows": [len(video_histories), len(meta_data)],
        "unique_video_ids": [
            video_histories["video_id"].nunique(dropna=True),
            meta_data["id"].nunique(dropna=True),
        ],
    }
)

input_summary


## Validate Required Columns

Before joining, we check that the two upstream notebooks produced the expected fields. This makes the notebook fail early if a preceding cleaning or scraping step changed its schema.

In [ ]:
def ensure_columns(dataframe, required_columns, label):
    missing_columns = sorted(set(required_columns) - set(dataframe.columns))
    if missing_columns:
        raise ValueError(
            f"Missing columns in {label}: {missing_columns}. "
            f"Found: {list(dataframe.columns)}"
        )


required_watch_columns = {
    "Participant ID",
    "time",
    "video_id",
    "watched_title",
    "clean_title",
    "url",
}

required_metadata_columns = {
    "id",
    "title",
    "duration",
    "channel_id",
    "channel",
    "aspect_ratio",
    "width",
    "height",
    "error",
    "metadata_status",
}

ensure_columns(video_histories, required_watch_columns, "video_histories.csv")
ensure_columns(meta_data, required_metadata_columns, "meta_data.csv")

print("Required columns are present.")

## Prepare Watch Events

The watch table already contains one row per validated video watch. Here we parse time, clean the join key, and rename channel fields from the donation so they do not get confused with metadata channel fields.

In [ ]:
video_histories = video_histories.copy()
watch_event_count = len(video_histories)
unique_watched_video_count = video_histories["video_id"].nunique(dropna=True)

video_histories["Participant ID"] = video_histories["Participant ID"].astype("string").str.strip()
video_histories["video_id"] = video_histories["video_id"].astype("string").str.strip()
video_histories["time"] = pd.to_datetime(
    video_histories["time"],
    utc=True,
    format="mixed",
    errors="coerce",
)

video_histories = video_histories.rename(
    columns={
        "channel_title": "donated_channel_title",
        "channel_url": "donated_channel_url",
        "channel_id": "donated_channel_id",
    }
)

if video_histories["time"].isna().any():
    raise ValueError("Some watch events have invalid timestamps.")

valid_video_id = video_histories["video_id"].str.match(r"^[A-Za-z0-9_-]{11}$", na=False)
if not valid_video_id.all():
    invalid_examples = video_histories.loc[~valid_video_id, "video_id"].head(10).tolist()
    raise ValueError(f"Some watch events have invalid video_id values: {invalid_examples}")

video_histories.head()

## Prepare And Deduplicate Metadata

Scraping pipelines can produce duplicate metadata rows when several scrape batches are combined. Following the original metadata-combination logic, we keep one row per video ID and prefer rows with usable metadata: successful status, title, duration, and channel ID.

In [ ]:
def has_text(series):
    return series.notna() & series.astype("string").str.strip().ne("").fillna(False)


meta_data = meta_data.copy()
metadata_rows_before_deduplication = len(meta_data)
metadata_duplicate_id_rows = int(meta_data.duplicated("id").sum())

meta_data["id"] = meta_data["id"].astype("string").str.strip()
meta_data["metadata_status"] = meta_data["metadata_status"].astype("string").str.strip().str.lower()

for numeric_column in [
    "duration",
    "view_count",
    "like_count",
    "comment_count",
    "channel_follower_count",
    "timestamp",
    "width",
    "height",
    "fps",
    "aspect_ratio",
]:
    if numeric_column in meta_data.columns:
        meta_data[numeric_column] = pd.to_numeric(meta_data[numeric_column], errors="coerce")

if "upload_date" in meta_data.columns:
    meta_data["upload_date"] = meta_data["upload_date"].astype("string").str.replace(r"\.0$", "", regex=True)

metadata_with_id = meta_data[has_text(meta_data["id"])].copy()
metadata_rows_without_id = len(meta_data) - len(metadata_with_id)

metadata_with_id["_status_priority"] = metadata_with_id["metadata_status"].eq("ok").fillna(False).astype(int)
metadata_with_id["_title_priority"] = has_text(metadata_with_id["title"]).astype(int)
metadata_with_id["_duration_priority"] = metadata_with_id["duration"].notna().astype(int)
metadata_with_id["_channel_priority"] = has_text(metadata_with_id["channel_id"]).astype(int)

metadata_deduplicated = (
    metadata_with_id
    .sort_values(
        [
            "id",
            "_status_priority",
            "_title_priority",
            "_duration_priority",
            "_channel_priority",
        ],
        ascending=[True, False, False, False, False],
        kind="mergesort",
    )
    .drop_duplicates(subset=["id"], keep="first")
    .drop(
        columns=[
            "_status_priority",
            "_title_priority",
            "_duration_priority",
            "_channel_priority",
        ]
    )
)

metadata_deduplicated.shape

## Preserve Metadata Errors

The scraper stores raw `yt_dlp` error messages because they often contain valuable information: private video, age restriction, account termination, copyright block, country block, subtitle download failure, or other availability signals.

The raw message is kept unchanged as `metadata_error`. The next step creates separate normalized and manually classified fields, so readers can inspect both the original scrape message and the analytical category derived from it.


In [ ]:
# Keep raw scrape errors unchanged. Missing errors stay missing.
metadata_deduplicated["error"] = metadata_deduplicated["error"].where(
    metadata_deduplicated["error"].notna(),
    pd.NA,
)

metadata_deduplicated[["id", "metadata_status", "error"]].head()

## Classify Metadata Errors

The classification has two layers. First, raw scraper errors are normalized into generalized messages so that video IDs, creator names, and repeated `yt_dlp` prefixes do not create artificial categories. Second, the generalized message is matched to the manual classification table from the original workflow.

If a future scrape creates a generalized message that is not in the manual table or the explicit alias map, the notebook stops and prints the missing messages. This is intentional: new availability messages should be reviewed rather than silently assigned.


In [ ]:
manual_map = pd.read_csv(MANUAL_CLASSIFICATION_PATH, dtype=str, low_memory=False)
manual_map = manual_map.rename(
    columns={
        "message": "generalized_error_msg",
        "classification": "category",
        "Critical": "critical",
    }
)
ensure_columns(
    manual_map,
    {"generalized_error_msg", "category", "critical"},
    "manual classification CSV",
)
manual_map = manual_map[["generalized_error_msg", "category", "critical"]].copy()
manual_map["generalized_error_msg"] = manual_map["generalized_error_msg"].astype(str).str.strip()
manual_map["category"] = manual_map["category"].fillna("").astype(str).str.strip()
manual_map["critical"] = manual_map["critical"].fillna("").astype(str).str.strip()

if manual_map["generalized_error_msg"].duplicated().any():
    duplicates = manual_map.loc[
        manual_map["generalized_error_msg"].duplicated(),
        "generalized_error_msg",
    ].unique().tolist()
    raise ValueError(f"Duplicate generalized errors in manual classification file: {duplicates[:10]}")

error_source = metadata_deduplicated[["id", "error"]].rename(
    columns={"id": "video_id", "error": "raw_error"}
).copy()
error_source["video_id"] = error_source["video_id"].astype("string").str.strip()
error_source["raw_error"] = error_source["raw_error"].where(has_text(error_source["raw_error"]), pd.NA)

errored_videos = error_source[has_text(error_source["raw_error"])].copy()
errored_videos["generalized_error_msg"] = errored_videos.apply(
    lambda row: normalize_error_message(row["raw_error"], row["video_id"]),
    axis=1,
)

errored_videos["classification_message"] = errored_videos["generalized_error_msg"].replace(
    ERROR_CLASSIFICATION_MESSAGE_ALIASES
)
alias_applied_count = int(
    errored_videos["classification_message"].ne(errored_videos["generalized_error_msg"]).sum()
)
errored_videos["generalized_error_msg"] = errored_videos["classification_message"]
errored_videos = errored_videos.drop(columns=["classification_message"])

missing_messages = sorted(
    set(errored_videos["generalized_error_msg"]) - set(manual_map["generalized_error_msg"])
)
if missing_messages:
    missing_error_messages = pd.DataFrame({"generalized_error_msg": missing_messages})
    display(missing_error_messages)
    raise ValueError(
        "The manual classification table is missing generalized error messages. "
        "Review the table above and add the messages before continuing."
    )

classified_errors = errored_videos.merge(
    manual_map,
    on="generalized_error_msg",
    how="left",
    validate="m:1",
)
classified_errors = classified_errors.drop_duplicates(subset=["video_id"]).copy()

error_types_by_video = error_source[["video_id"]].drop_duplicates().merge(
    classified_errors[
        ["video_id", "raw_error", "generalized_error_msg", "category", "critical"]
    ],
    on="video_id",
    how="left",
    validate="1:1",
)

error_types_by_video["category"] = error_types_by_video["category"].fillna("No error")
error_types_by_video["critical"] = error_types_by_video["critical"].fillna("").astype(str).str.strip()
error_types_by_video["generalized_error_msg"] = error_types_by_video["generalized_error_msg"].fillna("No error")
error_types_by_video["assessment_label"] = (
    error_types_by_video["critical"].map(ASSESSMENT_MAP).fillna("No error")
)
error_types_by_video["is_problematic"] = error_types_by_video["critical"].str.upper().eq("Y")

terminated_mask = error_types_by_video["generalized_error_msg"].eq(ACCOUNT_TERMINATED_MESSAGE)
terminated_critical = "Y" if ACCOUNT_TERMINATED_IS_PROBLEMATIC else "N"
terminated_label = "Problematic" if ACCOUNT_TERMINATED_IS_PROBLEMATIC else "Not problematic"
error_types_by_video.loc[terminated_mask, "critical"] = terminated_critical
error_types_by_video.loc[terminated_mask, "assessment_label"] = terminated_label
error_types_by_video.loc[terminated_mask, "is_problematic"] = ACCOUNT_TERMINATED_IS_PROBLEMATIC

error_types_by_video = error_types_by_video[
    [
        "video_id",
        "raw_error",
        "generalized_error_msg",
        "category",
        "critical",
        "assessment_label",
        "is_problematic",
    ]
].sort_values("video_id").reset_index(drop=True)

if error_types_by_video["video_id"].duplicated().any():
    raise AssertionError("Expected one row per video_id in error_types_by_video.")

error_classification_summary = pd.DataFrame(
    {
        "measure": [
            "metadata_videos",
            "metadata_videos_with_raw_error",
            "unique_generalized_error_messages",
            "alias_applied_videos",
            "unclassified_generalized_messages",
            "problematic_metadata_videos",
            "terminated_account_videos",
        ],
        "value": [
            len(error_types_by_video),
            int(error_types_by_video["raw_error"].notna().sum()),
            int(error_types_by_video.loc[error_types_by_video["raw_error"].notna(), "generalized_error_msg"].nunique()),
            alias_applied_count,
            len(missing_messages),
            int(error_types_by_video["is_problematic"].sum()),
            int(terminated_mask.sum()),
        ],
    }
)

error_classification_summary


## Rename Metadata Fields

Several fields exist in both donation data and metadata. Prefixing metadata fields keeps the source of each column clear after the join.

In [ ]:
metadata_rename_map = {
    "id": "video_id",
    "title": "metadata_title",
    "channel_id": "metadata_channel_id",
    "channel": "metadata_channel",
    "uploader": "metadata_uploader",
    "duration": "duration_seconds",
    "timestamp": "metadata_timestamp",
    "webpage_url": "metadata_url",
    "error": "metadata_error",
}

metadata_for_join = metadata_deduplicated.rename(columns=metadata_rename_map)

metadata_columns_to_keep = [
    "video_id",
    "metadata_title",
    "metadata_channel_id",
    "metadata_channel",
    "metadata_uploader",
    "duration_seconds",
    "view_count",
    "like_count",
    "comment_count",
    "channel_follower_count",
    "upload_date",
    "metadata_timestamp",
    "metadata_url",
    "categories",
    "tags",
    "language",
    "resolution",
    "width",
    "height",
    "fps",
    "aspect_ratio",
    "metadata_status",
    "metadata_error",
]

metadata_columns_to_keep = [
    column for column in metadata_columns_to_keep
    if column in metadata_for_join.columns
]

metadata_for_join = metadata_for_join[metadata_columns_to_keep].copy()

metadata_for_join.head()

## Join Watch Events To Metadata

The join is a left join from watch history to metadata. That means no donated watch events are dropped. If metadata is missing or errored, the event remains in the output and the linkage columns make that visible.

In [ ]:
def optional_column(dataframe, column):
    if column in dataframe.columns:
        return dataframe[column]
    return pd.Series(pd.NA, index=dataframe.index)


video_histories_enriched = video_histories.merge(
    metadata_for_join,
    on="video_id",
    how="left",
    validate="m:1",
    indicator=True,
)

video_histories_enriched["has_metadata"] = video_histories_enriched["_merge"].eq("both")
metadata_error_has_text = has_text(video_histories_enriched["metadata_error"])
metadata_status_is_error = (
    video_histories_enriched["metadata_status"]
    .astype("string")
    .str.lower()
    .eq("error")
    .fillna(False)
)

video_histories_enriched["metadata_join_status"] = "ok"
video_histories_enriched.loc[
    ~video_histories_enriched["has_metadata"],
    "metadata_join_status",
] = "missing_metadata"
video_histories_enriched.loc[
    video_histories_enriched["has_metadata"] & (metadata_error_has_text | metadata_status_is_error),
    "metadata_join_status",
] = "metadata_error"

video_histories_enriched["has_duration"] = video_histories_enriched["duration_seconds"].notna()

video_histories_enriched["analysis_channel_id"] = video_histories_enriched["metadata_channel_id"].combine_first(
    optional_column(video_histories_enriched, "donated_channel_id")
)
video_histories_enriched["analysis_channel_title"] = (
    video_histories_enriched["metadata_channel"]
    .combine_first(optional_column(video_histories_enriched, "metadata_uploader"))
    .combine_first(optional_column(video_histories_enriched, "donated_channel_title"))
)
video_histories_enriched["has_channel_id"] = has_text(video_histories_enriched["analysis_channel_id"])

video_histories_enriched = video_histories_enriched.drop(columns=["_merge"])

video_histories_enriched.shape

## Add Problematic-View Labels

The video-level error catalog is joined back onto the event-level watch table. This keeps the unit of analysis as watch events while using one stable classification per unique video ID. The raw `metadata_error` column remains unchanged; the added columns show the normalized message and manual assessment.


In [ ]:
problematic_labels_for_join = error_types_by_video.rename(
    columns={
        "category": "problematic_category",
        "critical": "problematic_critical",
        "assessment_label": "problematic_assessment_label",
    }
)[
    [
        "video_id",
        "generalized_error_msg",
        "problematic_category",
        "problematic_critical",
        "problematic_assessment_label",
        "is_problematic",
    ]
]

video_histories_enriched = video_histories_enriched.merge(
    problematic_labels_for_join,
    on="video_id",
    how="left",
    validate="m:1",
)

missing_problematic_lookup = video_histories_enriched["generalized_error_msg"].isna()
if missing_problematic_lookup.any():
    video_histories_enriched.loc[missing_problematic_lookup, "generalized_error_msg"] = "Missing metadata"
    video_histories_enriched.loc[missing_problematic_lookup, "problematic_category"] = "Missing metadata"
    video_histories_enriched.loc[missing_problematic_lookup, "problematic_critical"] = ""
    video_histories_enriched.loc[missing_problematic_lookup, "problematic_assessment_label"] = "Missing metadata"
    video_histories_enriched.loc[missing_problematic_lookup, "is_problematic"] = False

video_histories_enriched["is_problematic"] = video_histories_enriched["is_problematic"].astype("boolean")

metadata_error_event_mask = video_histories_enriched["metadata_join_status"].eq("metadata_error")
if video_histories_enriched.loc[metadata_error_event_mask, "problematic_category"].isna().any():
    raise AssertionError("Some metadata-error watch events are missing problematic-view categories.")

video_histories_enriched[
    [
        "video_id",
        "metadata_error",
        "generalized_error_msg",
        "problematic_category",
        "problematic_critical",
        "problematic_assessment_label",
        "is_problematic",
    ]
].head()


## Add Shorts/Longs And Day/Night Flags

These are full-dataset event labels, not period summaries. The timestamp stays in UTC as `time`; local-time helper columns are added only so daytime and nighttime can be interpreted in the configured timezone.

Shorts are identified with the same metadata rule used in the original workflow: vertical or square-ish videos with duration at most 180 seconds. Missing Shorts evidence is treated as not short, matching the original downstream handling.

In [ ]:
video_histories_enriched["watch_local_time"] = video_histories_enriched["time"].dt.tz_convert(ANALYSIS_TIMEZONE)
video_histories_enriched["watch_local_date"] = video_histories_enriched["watch_local_time"].dt.strftime("%Y-%m-%d")
video_histories_enriched["watch_local_hour"] = video_histories_enriched["watch_local_time"].dt.hour

is_daytime = hour_is_in_interval(
    video_histories_enriched["watch_local_hour"],
    DAYTIME_START_HOUR,
    DAYTIME_END_HOUR,
)
video_histories_enriched["is_daytime_watch"] = is_daytime.astype("boolean")
video_histories_enriched["is_nighttime_watch"] = (~is_daytime).astype("boolean")
video_histories_enriched["time_of_day"] = pd.Series("night", index=video_histories_enriched.index, dtype="string")
video_histories_enriched.loc[video_histories_enriched["is_daytime_watch"], "time_of_day"] = "day"

video_histories_enriched["shorts_metadata_available"] = (
    video_histories_enriched["aspect_ratio"].notna()
    & video_histories_enriched["duration_seconds"].notna()
)
video_histories_enriched["is_short"] = (
    video_histories_enriched["shorts_metadata_available"]
    & video_histories_enriched["aspect_ratio"].le(SHORT_MAX_ASPECT_RATIO)
    & video_histories_enriched["duration_seconds"].le(SHORT_MAX_DURATION_SECONDS)
).astype("boolean")
video_histories_enriched["is_long"] = (~video_histories_enriched["is_short"]).astype("boolean")

# Backward-compatible alias for readers who saw the earlier metadata-link notebook.
video_histories_enriched["is_short_by_metadata"] = video_histories_enriched["is_short"]

if not (video_histories_enriched["is_daytime_watch"] ^ video_histories_enriched["is_nighttime_watch"]).all():
    raise AssertionError("Each event should be exactly one of daytime or nighttime.")
if not (video_histories_enriched["is_short"] ^ video_histories_enriched["is_long"]).all():
    raise AssertionError("Each event should be exactly one of short or long.")

event_flag_diagnostics = pd.DataFrame(
    {
        "measure": [
            "analysis_timezone",
            "daytime_start_hour",
            "daytime_end_hour",
            "short_max_duration_seconds",
            "short_max_aspect_ratio",
            "shorts_watch_events",
            "longs_watch_events",
            "watch_events_missing_shorts_metadata",
            "daytime_watch_events",
            "nighttime_watch_events",
        ],
        "value": [
            ANALYSIS_TIMEZONE,
            DAYTIME_START_HOUR,
            DAYTIME_END_HOUR,
            SHORT_MAX_DURATION_SECONDS,
            SHORT_MAX_ASPECT_RATIO,
            int(video_histories_enriched["is_short"].sum()),
            int(video_histories_enriched["is_long"].sum()),
            int((~video_histories_enriched["shorts_metadata_available"]).sum()),
            int(video_histories_enriched["is_daytime_watch"].sum()),
            int(video_histories_enriched["is_nighttime_watch"].sum()),
        ],
    }
)

event_flag_diagnostics


## Add Search-Associated Watch Marker

The original full-data workflow estimated a post-search interval from the distribution of search-watch distances. Here we do not refit that model on mock data. Instead, we apply the validated interval from the original analysis as a fixed methodological parameter: a watch is marked as searched when a participant search occurred 1 to 55 seconds before the watch event.

Only search timestamps and participant IDs are used. Search query text is intentionally not copied into the enriched watch table.

In [ ]:
SEARCH_INTERVALS_SEC = [(1, 55)]

search_query_histories = pd.read_csv(
    SEARCH_QUERY_HISTORIES_PATH,
    usecols=["Participant ID", "time"],
    dtype={"Participant ID": "string"},
    low_memory=False,
)
ensure_columns(search_query_histories, {"Participant ID", "time"}, "search_query_histories.csv")

search_query_histories["Participant ID"] = search_query_histories["Participant ID"].astype("string").str.strip()
search_query_histories["time"] = pd.to_datetime(
    search_query_histories["time"],
    utc=True,
    format="mixed",
    errors="coerce",
)
search_query_histories = search_query_histories.dropna(subset=["Participant ID", "time"]).copy()
search_participant_ids = set(search_query_histories["Participant ID"].astype(str))

video_histories_enriched["has_search_history"] = (
    video_histories_enriched["Participant ID"].astype(str).isin(search_participant_ids)
)
video_histories_enriched["searched_for"] = pd.Series(pd.NA, index=video_histories_enriched.index, dtype="boolean")
video_histories_enriched.loc[video_histories_enriched["has_search_history"], "searched_for"] = False
video_histories_enriched["seconds_since_prior_search"] = pd.NA

watch_lookup = video_histories_enriched[["Participant ID", "time"]].copy()
watch_lookup["watch_event_position"] = video_histories_enriched.index

matched_search_parts = []
search_lookup = search_query_histories[["Participant ID", "time"]].rename(columns={"time": "search_time"})

for left_seconds, right_seconds in SEARCH_INTERVALS_SEC:
    if left_seconds < 0 or right_seconds < left_seconds:
        raise ValueError(f"Invalid search interval: {(left_seconds, right_seconds)}")

    interval_lookup = watch_lookup.copy()
    interval_lookup["search_lookup_time"] = interval_lookup["time"] - pd.to_timedelta(left_seconds, unit="s")

    participant_matches = []
    for participant_id, participant_watches in interval_lookup.groupby("Participant ID", sort=False):
        participant_searches = (
            search_lookup.loc[search_lookup["Participant ID"].eq(participant_id), ["search_time"]]
            .sort_values("search_time")
        )
        participant_watches = participant_watches.sort_values("search_lookup_time")

        if participant_searches.empty:
            participant_watches["search_time"] = pd.NaT
        else:
            participant_watches = pd.merge_asof(
                participant_watches,
                participant_searches,
                left_on="search_lookup_time",
                right_on="search_time",
                direction="backward",
                tolerance=pd.to_timedelta(right_seconds - left_seconds, unit="s"),
            )
        participant_matches.append(participant_watches)

    interval_matches = pd.concat(participant_matches, ignore_index=True)
    interval_matches["seconds_since_prior_search"] = (
        interval_matches["time"] - interval_matches["search_time"]
    ).dt.total_seconds()
    interval_matches = interval_matches[interval_matches["search_time"].notna()].copy()
    matched_search_parts.append(interval_matches)

if matched_search_parts:
    matched_searches = pd.concat(matched_search_parts, ignore_index=True)
    matched_searches = (
        matched_searches
        .sort_values(["watch_event_position", "seconds_since_prior_search"])
        .drop_duplicates(subset=["watch_event_position"], keep="first")
    )
    matched_positions = matched_searches["watch_event_position"].astype(int)
    video_histories_enriched.loc[matched_positions, "searched_for"] = True
    video_histories_enriched.loc[matched_positions, "seconds_since_prior_search"] = (
        matched_searches.set_index("watch_event_position")["seconds_since_prior_search"]
    )
else:
    matched_searches = pd.DataFrame(
        columns=["watch_event_position", "search_time", "seconds_since_prior_search"]
    )

search_diagnostics = pd.DataFrame(
    {
        "measure": [
            "search_query_rows",
            "participants_with_search_history",
            "search_intervals_applied",
            "searched_watch_events",
            "not_searched_watch_events",
            "watch_events_without_search_history",
        ],
        "value": [
            len(search_query_histories),
            len(search_participant_ids),
            str(SEARCH_INTERVALS_SEC),
            int(video_histories_enriched["searched_for"].eq(True).sum()),
            int(video_histories_enriched["searched_for"].eq(False).sum()),
            int(video_histories_enriched["searched_for"].isna().sum()),
        ],
    }
)

search_diagnostics


## Add Subscribed-Channel Marker

The subscription marker follows the original method: compare each watched video's metadata-derived `metadata_channel_id` to the participant's donated subscription channel IDs.

The enriched table already contains `analysis_channel_id`, which can fall back to channel information from the donated watch history. We keep that field for later sensitivity checks, but the strict `subscribed` marker below uses only `metadata_channel_id` to match the original workflow.

In [ ]:
subscription_files = sorted(
    MOCK_TAKEOUT_DIR.glob("*/YouTube and YouTube Music/subscriptions/subscriptions.csv")
)

subscription_frames = []
for path in subscription_files:
    participant_id = path.relative_to(MOCK_TAKEOUT_DIR).parts[0]
    subscriptions = pd.read_csv(path, dtype="string", low_memory=False)
    subscriptions["Participant ID"] = participant_id
    subscription_frames.append(subscriptions)

if subscription_frames:
    subscription_histories = pd.concat(subscription_frames, ignore_index=True)
else:
    subscription_histories = pd.DataFrame(
        columns=["Participant ID", "Channel Id", "Channel Url", "Channel Title"]
    )

subscription_histories = subscription_histories.rename(
    columns={
        "Channel Id": "subscribed_channel_id",
        "Channel Url": "subscribed_channel_url",
        "Channel Title": "subscribed_channel_title",
    }
)
ensure_columns(
    subscription_histories,
    {"Participant ID", "subscribed_channel_id", "subscribed_channel_url", "subscribed_channel_title"},
    "subscription histories",
)

subscription_histories = subscription_histories[
    ["Participant ID", "subscribed_channel_id", "subscribed_channel_url", "subscribed_channel_title"]
].copy()
subscription_histories["Participant ID"] = subscription_histories["Participant ID"].astype("string").str.strip()
subscription_histories["subscribed_channel_id"] = subscription_histories["subscribed_channel_id"].astype("string").str.strip()
subscription_histories = subscription_histories.dropna(subset=["Participant ID", "subscribed_channel_id"]).drop_duplicates()

subscription_participant_ids = set(subscription_histories["Participant ID"].astype(str))
subscribed_channel_pairs = set(
    zip(
        subscription_histories["Participant ID"].astype(str),
        subscription_histories["subscribed_channel_id"].astype(str),
    )
)

video_histories_enriched["has_subscription_history"] = (
    video_histories_enriched["Participant ID"].astype(str).isin(subscription_participant_ids)
)
video_histories_enriched["subscribed"] = pd.Series(pd.NA, index=video_histories_enriched.index, dtype="boolean")

has_metadata_channel_id = has_text(video_histories_enriched["metadata_channel_id"])
can_check_subscription = video_histories_enriched["has_subscription_history"] & has_metadata_channel_id

video_histories_enriched.loc[can_check_subscription, "subscribed"] = [
    (str(participant_id), str(channel_id)) in subscribed_channel_pairs
    for participant_id, channel_id in zip(
        video_histories_enriched.loc[can_check_subscription, "Participant ID"],
        video_histories_enriched.loc[can_check_subscription, "metadata_channel_id"],
    )
]

subscription_diagnostics = pd.DataFrame(
    {
        "measure": [
            "subscription_files",
            "subscription_rows",
            "participants_with_subscription_history",
            "subscribed_watch_events",
            "not_subscribed_watch_events",
            "watch_events_without_subscription_status",
            "watch_events_missing_metadata_channel_id",
        ],
        "value": [
            len(subscription_files),
            len(subscription_histories),
            len(subscription_participant_ids),
            int(video_histories_enriched["subscribed"].eq(True).sum()),
            int(video_histories_enriched["subscribed"].eq(False).sum()),
            int(video_histories_enriched["subscribed"].isna().sum()),
            int((~has_metadata_channel_id).sum()),
        ],
    }
)

subscription_diagnostics


## Select The Handoff Columns

The handoff table keeps the columns needed for the next notebooks, while avoiding very large metadata text fields. The original `meta_data.csv` remains the place to inspect descriptions and subtitle text, and `error_types_by_video.csv` stores the video-level error classification catalog.


In [ ]:
preferred_output_columns = [
    "Participant ID",
    "time",
    "watch_local_time",
    "watch_local_date",
    "watch_local_hour",
    "time_of_day",
    "is_daytime_watch",
    "is_nighttime_watch",
    "video_id",
    "watched_title",
    "clean_title",
    "watch_action_prefix",
    "watch_action_language_hint",
    "url",
    "donated_channel_title",
    "donated_channel_url",
    "donated_channel_id",
    "metadata_title",
    "metadata_channel",
    "metadata_uploader",
    "metadata_channel_id",
    "analysis_channel_title",
    "analysis_channel_id",
    "duration_seconds",
    "aspect_ratio",
    "width",
    "height",
    "resolution",
    "fps",
    "shorts_metadata_available",
    "is_short",
    "is_long",
    "is_short_by_metadata",
    "view_count",
    "like_count",
    "comment_count",
    "channel_follower_count",
    "upload_date",
    "metadata_timestamp",
    "metadata_url",
    "categories",
    "tags",
    "language",
    "metadata_status",
    "metadata_join_status",
    "searched_for",
    "has_search_history",
    "seconds_since_prior_search",
    "subscribed",
    "has_subscription_history",
    "has_metadata",
    "has_duration",
    "has_channel_id",
    "metadata_error",
    "generalized_error_msg",
    "problematic_category",
    "problematic_critical",
    "problematic_assessment_label",
    "is_problematic",
]

output_columns = [
    column for column in preferred_output_columns
    if column in video_histories_enriched.columns
]

video_histories_enriched = video_histories_enriched[output_columns].copy()

video_histories_enriched.head()


## Build Linkage Diagnostics

These checks document what the join did. They are useful both for teaching and for catching methodological problems before later notebooks compute ratios.

In [ ]:
matched_events = video_histories_enriched["has_metadata"].sum()
missing_metadata_events = video_histories_enriched["metadata_join_status"].eq("missing_metadata").sum()
metadata_error_events = video_histories_enriched["metadata_join_status"].eq("metadata_error").sum()
missing_duration_events = (~video_histories_enriched["has_duration"]).sum()
missing_channel_id_events = (~video_histories_enriched["has_channel_id"]).sum()
shorts_candidate_events = video_histories_enriched["is_short"].eq(True).sum()

metadata_linkage_report = pd.DataFrame(
    [
        {"measure": "watch_events", "value": int(watch_event_count)},
        {"measure": "unique_watched_videos", "value": int(unique_watched_video_count)},
        {"measure": "metadata_rows_before_deduplication", "value": int(metadata_rows_before_deduplication)},
        {"measure": "metadata_rows_after_deduplication", "value": int(len(metadata_deduplicated))},
        {"measure": "metadata_duplicate_id_rows", "value": int(metadata_duplicate_id_rows)},
        {"measure": "metadata_rows_without_id", "value": int(metadata_rows_without_id)},
        {"measure": "matched_watch_events", "value": int(matched_events)},
        {"measure": "missing_metadata_watch_events", "value": int(missing_metadata_events)},
        {"measure": "metadata_error_watch_events", "value": int(metadata_error_events)},
        {"measure": "watch_events_missing_duration", "value": int(missing_duration_events)},
        {"measure": "watch_events_missing_channel_id", "value": int(missing_channel_id_events)},
        {"measure": "watch_events_flagged_short_by_metadata", "value": int(shorts_candidate_events)},
        {
            "measure": "matched_unique_videos",
            "value": int(video_histories_enriched.loc[video_histories_enriched["has_metadata"], "video_id"].nunique()),
        },
        {
            "measure": "missing_metadata_unique_videos",
            "value": int(video_histories_enriched.loc[video_histories_enriched["metadata_join_status"].eq("missing_metadata"), "video_id"].nunique()),
        },
        {
            "measure": "metadata_error_unique_videos",
            "value": int(video_histories_enriched.loc[video_histories_enriched["metadata_join_status"].eq("metadata_error"), "video_id"].nunique()),
        },
    ]
)

problematic_base_diagnostics = pd.DataFrame(
    [
        {
            "measure": "errored_unique_videos",
            "value": int(error_types_by_video.loc[error_types_by_video["raw_error"].notna(), "video_id"].nunique()),
        },
        {
            "measure": "unique_generalized_error_messages",
            "value": int(error_types_by_video.loc[error_types_by_video["raw_error"].notna(), "generalized_error_msg"].nunique()),
        },
        {"measure": "alias_applied_videos", "value": int(alias_applied_count)},
        {"measure": "unclassified_generalized_messages", "value": int(len(missing_messages))},
        {"measure": "problematic_watch_events", "value": int(video_histories_enriched["is_problematic"].sum())},
        {
            "measure": "problematic_unique_videos",
            "value": int(video_histories_enriched.loc[video_histories_enriched["is_problematic"].eq(True), "video_id"].nunique()),
        },
    ]
)

problematic_assessment_counts = pd.DataFrame(
    [
        {"measure": f"assessment_label_count::{label}", "value": int(count)}
        for label, count in video_histories_enriched["problematic_assessment_label"].value_counts(dropna=False).items()
    ]
)
problematic_category_counts = pd.DataFrame(
    [
        {"measure": f"category_count::{label}", "value": int(count)}
        for label, count in video_histories_enriched["problematic_category"].value_counts(dropna=False).items()
    ]
)
problematic_diagnostics = pd.concat(
    [problematic_base_diagnostics, problematic_assessment_counts, problematic_category_counts],
    ignore_index=True,
)

exposure_linkage_report = pd.concat(
    [
        search_diagnostics.assign(section="search"),
        subscription_diagnostics.assign(section="subscription"),
        event_flag_diagnostics.assign(section="event_flags"),
        problematic_diagnostics.assign(section="problematic_views"),
    ],
    ignore_index=True,
)[["section", "measure", "value"]]

metadata_linkage_report


## Save Outputs

The enriched table is the main handoff. The linkage and exposure reports are compact audit trails, while `error_types_by_video.csv` keeps the video-level classification catalog behind the event-level labels.


In [ ]:
if len(video_histories_enriched) != watch_event_count:
    raise AssertionError("The join changed the number of watch events.")

if video_histories_enriched["video_id"].isna().any():
    raise AssertionError("The enriched table contains missing video_id values.")

required_output_columns = {
    "duration_seconds",
    "aspect_ratio",
    "analysis_channel_id",
    "metadata_error",
    "searched_for",
    "has_search_history",
    "seconds_since_prior_search",
    "subscribed",
    "has_subscription_history",
    "watch_local_time",
    "watch_local_date",
    "watch_local_hour",
    "time_of_day",
    "is_daytime_watch",
    "is_nighttime_watch",
    "shorts_metadata_available",
    "is_short",
    "is_long",
    "generalized_error_msg",
    "problematic_category",
    "problematic_critical",
    "problematic_assessment_label",
    "is_problematic",
}
ensure_columns(video_histories_enriched, required_output_columns, "video_histories_enriched")
ensure_columns(
    error_types_by_video,
    {
        "video_id",
        "raw_error",
        "generalized_error_msg",
        "category",
        "critical",
        "assessment_label",
        "is_problematic",
    },
    "error_types_by_video",
)

unexpected_query_columns = sorted(
    set(video_histories_enriched.columns) & {"search_title", "query_from_title", "query_from_url"}
)
if unexpected_query_columns:
    raise AssertionError(f"Search query text leaked into the enriched table: {unexpected_query_columns}")

if error_types_by_video["video_id"].nunique(dropna=True) != len(error_types_by_video):
    raise AssertionError("error_types_by_video should contain one row per metadata video ID.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
video_histories_enriched.to_csv(ENRICHED_VIDEO_HISTORIES_PATH, index=False)
metadata_linkage_report.to_csv(METADATA_LINKAGE_REPORT_PATH, index=False)
subscription_histories.to_csv(SUBSCRIPTION_HISTORIES_PATH, index=False)
exposure_linkage_report.to_csv(EXPOSURE_LINKAGE_REPORT_PATH, index=False)
error_types_by_video.to_csv(ERROR_TYPES_BY_VIDEO_PATH, index=False)

for path in [
    ENRICHED_VIDEO_HISTORIES_PATH,
    METADATA_LINKAGE_REPORT_PATH,
    SUBSCRIPTION_HISTORIES_PATH,
    EXPOSURE_LINKAGE_REPORT_PATH,
    ERROR_TYPES_BY_VIDEO_PATH,
]:
    if not path.exists():
        raise AssertionError(f"Expected output was not created: {path.name}")

print(f"Saved {ENRICHED_VIDEO_HISTORIES_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Saved {METADATA_LINKAGE_REPORT_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Saved {SUBSCRIPTION_HISTORIES_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Saved {EXPOSURE_LINKAGE_REPORT_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Saved {ERROR_TYPES_BY_VIDEO_PATH.relative_to(PROJECT_ROOT).as_posix()}")


## Result

The diagnostics summarize the metadata join, exposure markers, event flags, and problematic-view classification. The table previews show the first five rows of each CSV stored by this notebook.


In [ ]:
print("Metadata diagnostics")
print(metadata_linkage_report.to_string(index=False))

print()
print("Exposure and classification diagnostics")
print(exposure_linkage_report.to_string(index=False))

print()
print("video_histories_enriched.csv")
display(video_histories_enriched.head(5))

print("metadata_linkage_report.csv")
display(metadata_linkage_report.head(5))

print("subscription_histories.csv")
display(subscription_histories.head(5))

print("exposure_linkage_report.csv")
display(exposure_linkage_report.head(5))

print("error_types_by_video.csv")
display(error_types_by_video.head(5))
